In [1]:
from pathlib import Path

ROOT = Path(
    "/Users/beck/Developer/Allstate-1C-predicting-auto-claims-severity"
)

SOURCE = ROOT / "data" / "allstate_claims_data.csv"
GATE1 = ROOT / "artifacts" / "october" / "gate 1"
MANIFEST = GATE1 / "october_partition_manifest.csv"
METADATA = GATE1 / "october_split_metadata.json"

for path in [SOURCE, MANIFEST, METADATA]:
    print(path.name, "exists:", path.is_file())

allstate_claims_data.csv exists: True
october_partition_manifest.csv exists: True
october_split_metadata.json exists: True


In [2]:
import json
import pandas as pd

manifest = pd.read_csv(MANIFEST)

with METADATA.open() as file:
    metadata = json.load(file)

print("Manifest columns:", manifest.columns.tolist())
print("\nActual partition counts:")
print(manifest["partition"].value_counts())

print("\nExpected partition counts:")
print(metadata["partition_counts"])

print("\nDuplicate IDs:", manifest["id"].duplicated().sum())
print("Missing values:", manifest.isna().sum().to_dict())

Manifest columns: ['id', 'partition']

Actual partition counts:
partition
train         141238
validation     28248
test           18832
Name: count, dtype: int64

Expected partition counts:
{'test': 18832, 'train': 141238, 'validation': 28248}

Duplicate IDs: 0
Missing values: {'id': 0, 'partition': 0}


In [3]:
import hashlib

def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


source_matches = (
    file_sha256(SOURCE) == metadata["source"]["sha256"]
)

manifest_matches = (
    file_sha256(MANIFEST)
    == metadata["artifacts_sha256"]["october_partition_manifest.csv"]
)

print("Source matches recorded version:", source_matches)
print("Manifest matches recorded version:", manifest_matches)

assert source_matches, "Source file differs from the recorded version."
assert manifest_matches, "Manifest differs from the recorded version."

Source matches recorded version: True
Manifest matches recorded version: True


In [4]:
categorical_cols = metadata["field_roles"]["categorical_predictors"]
continuous_cols = metadata["field_roles"]["continuous_predictors"]

feature_cols = categorical_cols + continuous_cols

forbidden_cols = {"id", "loss", "log1p_loss", "partition"}

assert categorical_cols == [f"cat{i}" for i in range(1, 117)]
assert continuous_cols == [f"cont{i}" for i in range(1, 15)]
assert len(set(feature_cols)) == 130
assert forbidden_cols.isdisjoint(feature_cols)

print("Categorical predictors:", len(categorical_cols))
print("Continuous predictors:", len(continuous_cols))
print("Total predictors:", len(feature_cols))
print("Forbidden columns included:", forbidden_cols.intersection(feature_cols))

Categorical predictors: 116
Continuous predictors: 14
Total predictors: 130
Forbidden columns included: set()


**STEP 2**

In [5]:
development_manifest = manifest.loc[
    manifest["partition"].isin(["train", "validation"])
]

development_ids = set(development_manifest["id"])

column_types = {
    "id": "int64",
    **{col: "string" for col in categorical_cols},
    **{col: "float64" for col in continuous_cols},
}

development_chunks = []

for chunk in pd.read_csv(
    SOURCE,
    usecols=["id"] + feature_cols,
    dtype=column_types,
    chunksize=10_000,
):
    selected = chunk.loc[chunk["id"].isin(development_ids)].copy()
    development_chunks.append(selected)

development = pd.concat(
    development_chunks, ignore_index=True
).set_index("id")

del development_chunks, chunk, selected

print("Development predictor shape:", development.shape)
print("Unique claim IDs:", development.index.is_unique)
print("Missing predictor values:", development.isna().sum().sum())
print("Columns match allowed inputs:", set(development.columns) == set(feature_cols))

Development predictor shape: (169486, 130)
Unique claim IDs: True
Missing predictor values: 0
Columns match allowed inputs: True


In [6]:
train_ids = manifest.loc[manifest["partition"].eq("train"), "id"]

valid_ids = manifest.loc[manifest["partition"].eq("validation"), "id"]

# Selects the specified claims in the manifest's order, using only the allowed columns. 
# This ensures that the training and validation sets are consistent with the manifest and that no forbidden columns are included.
X_train = development.loc[train_ids, feature_cols].copy()
X_valid = development.loc[valid_ids, feature_cols].copy()

# The assertions check that the sizes are correct, IDs remain aligned, the partitions don’t overlap, and forbidden columns are excluded.
assert X_train.shape == (141238, 130)
assert X_valid.shape == (28248, 130)

assert X_train.index.tolist() == train_ids.tolist()
assert X_valid.index.tolist() == valid_ids.tolist()

assert X_train.index.intersection(X_valid.index).empty

assert forbidden_cols.isdisjoint(X_train.columns)
assert forbidden_cols.isdisjoint(X_valid.columns)

# Only prints if every assertion passes to ensure that the training and validation sets are correctly prepared according to the manifest and metadata specifications.
print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("Claim order matches manifest: True")
print("No shared claims between training and validation: True")
print("Forbidden columns excluded: True")

Training shape: (141238, 130)
Validation shape: (28248, 130)
Claim order matches manifest: True
No shared claims between training and validation: True
Forbidden columns excluded: True


In [7]:
RARE_THRESHOLD = 30

# value_counts() counts each label within a field. 
# We then retain the labels whose counts are below 30. 
# Each field gets its own list: label "A" in cat1 is separate from label "A" in cat116.
training_category_counts = {
    col: X_train[col].value_counts()
    for col in categorical_cols
}

rare_levels = {
    col: counts.index[counts < RARE_THRESHOLD].tolist()
    for col, counts in training_category_counts.items()
}

rare_summary = pd.DataFrame({
    "training_levels": {
        col: len(counts)
        for col, counts in training_category_counts.items()
    },
    "rare_levels": {
        col: len(levels)
        for col, levels in rare_levels.items()
    },
})

print("Total rare levels:", rare_summary["rare_levels"].sum())
print("Fields with rare levels:", rare_summary["rare_levels"].gt(0).sum())

rare_summary.loc[rare_summary["rare_levels"].gt(0)].head(10)

Total rare levels: 352
Fields with rare levels: 30


,training_levels,rare_levels
cat15,2,1
cat22,2,1
cat70,2,1
cat73,3,1
cat74,3,1
cat88,4,1
cat89,8,5
cat90,7,3
cat91,8,1
cat92,7,3


In [8]:
example_col = "cat89"

example_counts = (
    training_category_counts[example_col]
    .rename_axis("original_label")
    .reset_index(name="training_count")
)

example_counts["is_rare"] = (
    example_counts["training_count"] < RARE_THRESHOLD
)

example_counts["grouped_label"] = example_counts["original_label"].where(
    ~example_counts["is_rare"],
    "__rare__",
)

example_counts

,original_label,training_count,is_rare,grouped_label
0,A,137816,False,A
1,B,3225,False,B
2,C,165,False,C
3,D,25,True,__rare__
4,E,3,True,__rare__
5,I,2,True,__rare__
6,H,1,True,__rare__
7,G,1,True,__rare__


In [9]:
rare_flags = pd.DataFrame({
    col: X_valid[col].isin(rare_levels[col])
    for col in categorical_cols
})

unseen_flags = pd.DataFrame({
    col: ~X_valid[col].isin(training_category_counts[col].index)
    for col in categorical_cols
})

print(
    "Validation claims with any training-rare label:",
    rare_flags.any(axis=1).sum(),
)

print(
    "Validation claims with any unseen label:",
    unseen_flags.any(axis=1).sum(),
)

print("Total unseen field occurrences:", unseen_flags.sum().sum())

unseen_by_field = unseen_flags.sum()
unseen_by_field.loc[unseen_by_field.gt(0)]

Validation claims with any training-rare label: 490
Validation claims with any unseen label: 12
Total unseen field occurrences: 13


cat105    1
cat107    1
cat109    1
cat110    1
cat114    1
cat116    8
dtype: int64

In [10]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version.split()[0])

for package in ["pandas", "numpy", "scikit-learn", "xgboost"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

Python: 3.13.2
pandas: 2.3.2
numpy: 2.3.3
scikit-learn: not installed
xgboost: not installed


In [11]:
%pip install scikit-learn==1.7.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 5.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin

print("scikit-learn:", sklearn.__version__)
print("Transformer imports successful")

scikit-learn: 1.7.2
Transformer imports successful


In [13]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=30):
        self.min_count = min_count

    def fit(self, X, y=None):
        if X.isna().any().any():
            raise ValueError("Training categories contain missing values.")

        if X.eq("__rare__").any().any():
            raise ValueError("The reserved label '__rare__' already exists.")

        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        self.known_levels_ = {}
        self.rare_levels_ = {}

        for col in self.feature_names_in_:
            counts = X[col].value_counts()

            self.known_levels_[col] = counts.index.tolist()
            self.rare_levels_[col] = counts.index[
                counts < self.min_count
            ].tolist()

        return self

In [14]:
category_grouper = RareCategoryGrouper(min_count=RARE_THRESHOLD)

category_grouper.fit(X_train[categorical_cols])

print("Categorical fields learned:", category_grouper.n_features_in_)
print(
    "Rare levels learned:",
    sum(len(levels) for levels in category_grouper.rare_levels_.values()),
)
print("Rare labels in cat89:", category_grouper.rare_levels_["cat89"])

Categorical fields learned: 116
Rare levels learned: 352
Rare labels in cat89: ['D', 'E', 'I', 'H', 'G']


In [15]:
category_grouper = RareCategoryGrouper(min_count=RARE_THRESHOLD)

train_categories = category_grouper.fit_transform(
    X_train[categorical_cols]
)

valid_categories = category_grouper.transform(
    X_valid[categorical_cols]
)

print("Training categories shape:", train_categories.shape)
print("Validation categories shape:", valid_categories.shape)

print("\nGrouped training counts for cat89:")
print(train_categories["cat89"].value_counts())

assert train_categories.index.equals(X_train.index)
assert valid_categories.index.equals(X_valid.index)

AttributeError: 'RareCategoryGrouper' object has no attribute 'transform'